# 03 — Sensor Performance Characterization

## Objective

Characterise a simulated detector from a flat-field illumination sweep: photon transfer curve (PTC), signal-to-noise vs. exposure, dynamic range, and linearity.

## What you'll see

- Flat-field captures at stepped illumination levels
- PTC (variance vs. mean) with a gain estimate, dynamic range, and linearity metrics
- `PTCAnalyzer`, `DynamicRangeAnalyzer`, `LinearityTestAnalyzer` outputs

## How to use this notebook

Change the parameters in the next cell and rerun the later cells to see how the metrics change.

## How to use this notebook

- Edit the parameters in the next cell to change the sweep shape, exposure, and detector noise.
- Run the cells in order.
- Compare how the same framework behaves for different settings.
- The later cells explain which analysis module is being used and what each metric means.

In [ ]:
# Editable parameters: change these values and rerun the notebook.
# levels: relative illumination values used in the flat-field sweep.
# exposure_time: detector integration time.
# gain: conversion gain from electrons to digital counts.
# read_noise_sigma: detector read noise in electrons.
# shape: image size for each captured frame.
levels = [0.1, 0.3, 0.5, 0.7, 1.0]
exposure_time = 1e-3
gain = 2.0
read_noise_sigma = 2.0
shape = (16, 16)

print("Configuration:")
print(f"  levels={levels}")
print(f"  exposure_time={exposure_time}")
print(f"  gain={gain}")
print(f"  read_noise_sigma={read_noise_sigma}")
print(f"  shape={shape}")

In [ ]:
import numpy as np

from optical_metrology.analysis import DynamicRangeAnalyzer, LinearityTestAnalyzer, PTCAnalyzer
from optical_metrology.detector import CMOSDetector
from optical_metrology.illumination import FlatFieldSource
from optical_metrology.optics import GaussianPSF, OpticalPropagator, OpticalSystem
from optical_metrology.scattering import LambertianScattering
from optical_metrology.surface import FlatSurface

In [ ]:
# Build the flat-field sweep and capture a set of simulated images.
# This step represents the sensor being illuminated at several known levels.
src = FlatFieldSource(wavelength=550e-9, power=1.0, intensity_levels=levels)
surf = FlatSurface(shape=shape)
scatter = LambertianScattering()
view = np.array([0.0, 0.0, 1.0])
optics = OpticalSystem(wavelength=550e-9, numerical_aperture=0.25, focal_length=50e-3, magnification=1.0)
propagator = OpticalPropagator(GaussianPSF(sigma=1.0), throughput_enabled=False)
detector = CMOSDetector(
    exposure_time=exposure_time,
    quantum_efficiency=0.5,
    gain=gain,
    read_noise_sigma=read_noise_sigma,
    rng_seed=42,
)

images = []
for light_field in src.generate_intensity_sweep(shape=shape, spacing=1.0):
    scattered = scatter.evaluate(light_field, surf, view)
    sensor_field = propagator.propagate(scattered, optics)
    images.append(detector.capture(sensor_field))

print(f"Captured {len(images)} images")
print("Each image corresponds to one illumination level in the sweep.")

In [ ]:
# Run the analysis modules on the captured images.
# PTCAnalyzer estimates gain and read noise from variance vs. mean.
# DynamicRangeAnalyzer summarizes the usable signal range of the last image.
# LinearityTestAnalyzer checks how closely the response follows a straight line.
ptc_result = PTCAnalyzer().analyze(images)
dynamic_range_result = DynamicRangeAnalyzer().analyze(images[-1])
linearity_result = LinearityTestAnalyzer(ideal_exposures=levels).analyze(images)

print("PTC summary:")
for key, value in ptc_result.measurements.items():
    print(f"  {key}: {value}")

print("\nDynamic range summary:")
for key, value in dynamic_range_result.measurements.items():
    print(f"  {key}: {value}")

print("\nLinearity summary:")
for key, value in linearity_result.measurements.items():
    print(f"  {key}: {value}")

## Try next

Try changing one thing at a time:
- increase the number of intensity levels to make the sweep more detailed
- change the exposure time or gain and see how the response curve shifts
- increase the read-noise level and observe how the PTC fit changes
- change the image size to see how the analysis scales with sensor resolution
- compare a low-noise detector with a noisier one by changing read_noise_sigma